# ACIS Insurance Risk Analytics: Hypothesis Testing

This notebook tests whether claim risk and profitability differ across key ACIS business segments.

The goal is to move beyond visual EDA and provide statistical evidence that ACIS leadership can use for pricing, underwriting, and marketing decisions.

## 1. Setup

Import reusable project utilities and set the statistical significance level. A result is considered statistically significant when `p_value < alpha`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_insurance_data, summarize_dataset
from src.eda_utils import add_risk_metrics, group_risk_summary
from src.hypothesis_tests import (
    build_hypothesis_summary,
    chi_square_test,
    claim_rate_z_test,
    independent_t_test,
)

ALPHA = 0.05
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

## 2. Load Cleaned Data

The notebook uses the cleaned dataset generated by the DVC `prepare_data` stage.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "insurance_data_cleaned.csv"

df = load_insurance_data(DATA_PATH)
df = add_risk_metrics(df)
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

summarize_dataset(df)

**Leadership note:** The analysis uses the cleaned DVC output so the statistical results can be reproduced from the versioned data pipeline.

## 3. Define Hypotheses and KPIs

Each hypothesis needs a clear KPI before selecting the statistical test.

| Business question | KPI | Test |
| --- | --- | --- |
| Does claim risk differ by province? | Claim occurrence rate (`HasClaim`) | Chi-square test |
| Does claim risk differ by gender? | Claim occurrence rate (`HasClaim`) | Chi-square test |
| Do two high-volume provinces differ in claim rate? | Claim occurrence rate | Two-proportion z-test |
| Do high-volume zip/postal code groups differ in margin? | Average margin | Welch's t-test |
| Do gender groups differ in average margin? | Average margin | Welch's t-test |

In [ ]:
def first_existing_column(columns: list[str], candidates: list[str]) -> str | None:
    """Return the first matching column name from a list of candidates."""
    return next((candidate for candidate in candidates if candidate in columns), None)


def top_two_groups(df: pd.DataFrame, group_col: str) -> list[str]:
    """Return the two largest groups by row count for stable comparisons."""
    counts = df[group_col].dropna().value_counts()
    if len(counts) < 2:
        raise ValueError(f"Column {group_col} needs at least two groups for comparison.")
    return counts.head(2).index.astype(str).tolist()


province_col = first_existing_column(df.columns.tolist(), ["Province", "province"])
gender_col = first_existing_column(df.columns.tolist(), ["Gender", "gender"])
zip_col = first_existing_column(
    df.columns.tolist(),
    ["PostalCode", "Postal_Code", "ZipCode", "ZIPCode", "Zip", "zip"],
)

province_col, gender_col, zip_col

## 4. Hypothesis 1: Claim Risk by Province

**Null hypothesis:** Claim occurrence is independent of province.

**Alternative hypothesis:** Claim occurrence differs by province.

**KPI:** Claim occurrence rate (`HasClaim`).

In [ ]:
results = []

if province_col:
    province_claim_result = chi_square_test(
        df,
        group_col=province_col,
        outcome_col="HasClaim",
        alpha=ALPHA,
        business_question="Does claim risk differ across provinces?",
    )
    results.append(province_claim_result)
    display(pd.DataFrame([province_claim_result]).drop(columns=["contingency_table", "expected_counts"]))
else:
    print("Province column not found; skipping province claim-risk test.")

**Leadership interpretation:** If the p-value is below 0.05, ACIS has statistical evidence that claim risk is not evenly distributed across provinces. This can support province-level pricing review or targeted underwriting actions.

## 5. Hypothesis 2: Claim Risk by Gender

**Null hypothesis:** Claim occurrence is independent of gender.

**Alternative hypothesis:** Claim occurrence differs by gender.

**KPI:** Claim occurrence rate (`HasClaim`).

In [ ]:
if gender_col:
    gender_claim_result = chi_square_test(
        df,
        group_col=gender_col,
        outcome_col="HasClaim",
        alpha=ALPHA,
        business_question="Does claim risk differ by gender?",
    )
    results.append(gender_claim_result)
    display(pd.DataFrame([gender_claim_result]).drop(columns=["contingency_table", "expected_counts"]))
else:
    print("Gender column not found; skipping gender claim-risk test.")

**Leadership interpretation:** Gender-level findings should be treated carefully and interpreted with other risk drivers such as geography, vehicle mix, policy type, and exposure. A significant result indicates association, not causation.

## 6. Hypothesis 3: Claim Rate Between Two High-Volume Provinces

**Null hypothesis:** The two selected provinces have equal claim rates.

**Alternative hypothesis:** The two selected provinces have different claim rates.

**KPI:** Claim occurrence rate.

The notebook automatically selects the two provinces with the largest sample sizes to keep the comparison stable.

In [ ]:
if province_col:
    province_a, province_b = top_two_groups(df, province_col)
    province_rate_result = claim_rate_z_test(
        df,
        group_col=province_col,
        group_a=province_a,
        group_b=province_b,
        alpha=ALPHA,
    )
    results.append(province_rate_result)
    display(pd.DataFrame([province_rate_result]))
else:
    print("Province column not found; skipping province claim-rate z-test.")

**Leadership interpretation:** This pairwise comparison helps ACIS understand whether high-volume provinces have materially different claim rates. If significant, the result can guide more focused province-level pricing and risk review.

## 7. Hypothesis 4: Margin by Zip or Postal Code

**Null hypothesis:** The selected zip/postal code groups have equal average margin.

**Alternative hypothesis:** The selected zip/postal code groups have different average margin.

**KPI:** Average margin (`TotalPremium - TotalClaims`).

In [ ]:
if zip_col:
    zip_a, zip_b = top_two_groups(df, zip_col)
    zip_margin_result = independent_t_test(
        df,
        group_col=zip_col,
        value_col="Margin",
        group_a=zip_a,
        group_b=zip_b,
        alpha=ALPHA,
        business_question=f"Does average margin differ between postal codes {zip_a} and {zip_b}?",
    )
    results.append(zip_margin_result)
    display(pd.DataFrame([zip_margin_result]))
else:
    print("Zip/postal code column not found; skipping zip-code margin test.")

**Leadership interpretation:** If margin differs significantly by zip or postal code, ACIS may have an opportunity to refine local pricing or marketing decisions. This should be reviewed alongside sample size and claim severity patterns.

## 8. Hypothesis 5: Margin by Gender

**Null hypothesis:** The selected gender groups have equal average margin.

**Alternative hypothesis:** The selected gender groups have different average margin.

**KPI:** Average margin (`TotalPremium - TotalClaims`).

In [ ]:
if gender_col:
    gender_a, gender_b = top_two_groups(df, gender_col)
    gender_margin_result = independent_t_test(
        df,
        group_col=gender_col,
        value_col="Margin",
        group_a=gender_a,
        group_b=gender_b,
        alpha=ALPHA,
        business_question=f"Does average margin differ between {gender_a} and {gender_b}?",
    )
    results.append(gender_margin_result)
    display(pd.DataFrame([gender_margin_result]))
else:
    print("Gender column not found; skipping gender margin test.")

**Leadership interpretation:** A statistically significant margin difference by gender would justify deeper analysis, but business action should consider fairness, regulation, and whether other variables explain the observed difference.

## 9. Summary Table

This table is designed for reporting. It keeps only the hypothesis, test used, p-value, decision, and business interpretation.

In [ ]:
summary = build_hypothesis_summary(results)

leadership_summary = summary.rename(
    columns={
        "business_question": "hypothesis",
        "test_name": "test_used",
        "reject_null": "decision",
        "interpretation": "business_interpretation",
    }
)

leadership_summary["decision"] = leadership_summary["decision"].map(
    {True: "Reject null", False: "Do not reject null"}
)

leadership_summary = leadership_summary[
    ["hypothesis", "test_used", "p_value", "decision", "business_interpretation"]
]

leadership_summary

## 10. Guidance for ACIS Leadership

Use the summary table as evidence for prioritizing deeper pricing and underwriting analysis.

- `Reject null` means there is statistical evidence that the tested KPI differs across the compared groups.
- `Do not reject null` means the data does not provide enough evidence of a difference at the chosen 5% significance level.
- Statistical significance does not automatically mean a pricing change should be made. ACIS should also consider business size, margin impact, regulatory constraints, and model-based feature importance.
- These results should feed into the final modeling stage, where multiple risk drivers can be evaluated together.

## 11. Save Report-Ready Results

Save the leadership summary so it can be reused directly in `reports/final_report.md` or submitted as supporting evidence.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

leadership_summary.to_csv(REPORTS_DIR / "hypothesis_test_results.csv", index=False)

rejected_hypotheses = leadership_summary[
    leadership_summary["decision"] == "Reject null"
]

rejected_hypotheses[["hypothesis", "business_interpretation"]]

**Final report note:** Focus the narrative on rejected hypotheses because they provide statistical evidence of segment differences. For non-rejected hypotheses, state that the available data does not provide enough evidence for a difference at the selected significance level.